# Notebook 06 — Motor de Decisión de Marketing: ROI y Acciones de Retención

**Inputs:** `customer_360.parquet` (NB03) + `churn_predictions.parquet` (NB05)  
**Objetivo:** Convertir scores de churn y CLV en decisiones de negocio accionables: qué acción de marketing aplicar a cada cliente y qué ROI esperado tiene.

---
### Pipeline
1. Carga y unificación de datos (RFM + churn + CLV)
2. Catálogo de acciones de marketing y modelo de costes
3. Cálculo de ROI esperado por acción y cliente
4. Motor de decisión: asignación óptima de acción por cliente
5. Simulación de presupuesto y priorización
6. Generación de mensajes personalizados
7. Panel ejecutivo de impacto esperado
8. Exportar plan de acción

## 0 · Setup

In [ ]:
import os
from pathlib import Path

notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    os.chdir(notebook_dir.parent)
print('Working directory:', Path.cwd())

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

PALETTE = {
    'primary':   '#2563EB',
    'secondary': '#7C3AED',
    'success':   '#10B981',
    'danger':    '#EF4444',
    'warning':   '#F59E0B',
    'neutral':   '#6B7280',
}
sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

DATA_DIR   = Path('data/processed')
OUTPUT_DIR = Path('outputs/06_marketing')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Setup completo.')

## 1 · Carga y unificación de datos

In [ ]:
c360 = pd.read_parquet(DATA_DIR / 'customer_360.parquet')

churn_path = DATA_DIR / 'churn_predictions.parquet'
if churn_path.exists():
    churn_df = pd.read_parquet(churn_path)
    # Unir solo columnas nuevas de churn
    new_cols = [c for c in churn_df.columns if c not in c360.columns or c == 'Customer ID']
    df = c360.merge(churn_df[new_cols], on='Customer ID', how='left')
else:
    raise FileNotFoundError(
        'No se encuentra churn_predictions.parquet. Ejecuta primero el Notebook 05.'
    )

print(f'Clientes: {len(df):,}')
print(f'Columnas: {list(df.columns)}')
df[['Customer ID','cluster_label','RFM_segment','clv_12m','churn_prob','risk_segment']].head()

In [ ]:
# ── Limpieza: clientes sin churn_prob o clv_12m no pueden recibir decisión ───
before = len(df)
df = df.dropna(subset=['churn_prob']).copy()
df['clv_12m'] = df['clv_12m'].fillna(df['monetary'])  # fallback si no hay CLV
print(f'Clientes con churn_prob válido: {len(df):,} / {before:,}')

## 2 · Simulación de A/B testing histórico y modelo de uplift

Las tasas de éxito de las campañas de marketing (`success_rate`) no son un número fijo igual para todos los clientes — dependen del perfil de cada cliente. Un cliente inactivo desde hace meses responde de forma distinta a un descuento agresivo que uno que compró la semana pasada.

Para capturar esta heterogeneidad, simulamos una campaña A/B histórica sobre los propios clientes del dataset y entrenamos un modelo de **uplift** que aprende el efecto esperado de cada acción condicionado al perfil RFM de cada cliente. **Los datos de campaña son sintéticos** — generados deliberadamente para poder demostrar el pipeline completo. En producción se sustituirían por el histórico real de campañas A/B de la empresa, sin cambiar ninguna otra parte del código.

**Cómo se simula la campaña:** a cada cliente se le asigna aleatoriamente un grupo (control o uno de los cuatro tratamientos). La respuesta (`responded`, 0/1) se genera con una probabilidad base por tratamiento más un efecto dependiente de sus métricas RFM — los clientes más inactivos (recency alta) responden mejor a incentivos agresivos, y los muy frecuentes tienen menos margen de mejora. Se añade ruido aleatorio para que la relación no sea determinista.

In [ ]:
rng = np.random.default_rng(42)

TREATMENTS = ['control', 'newsletter', 'email_personalizado', 'descuento_agresivo', 'llamada_personal']

# Asignación aleatoria de tratamiento (A/B/C/D/E test)
ab_df = df[['Customer ID','recency','frequency','monetary','avg_order_value','churn_prob']].copy()
ab_df['treatment'] = rng.choice(TREATMENTS, size=len(ab_df), p=[0.2,0.2,0.2,0.2,0.2])

# Features normalizadas para construir la probabilidad de respuesta
ab_df['recency_norm']   = (ab_df['recency'] - ab_df['recency'].mean()) / ab_df['recency'].std()
ab_df['frequency_norm'] = (np.log1p(ab_df['frequency']) - np.log1p(ab_df['frequency']).mean()) / np.log1p(ab_df['frequency']).std()

# Probabilidad base por tratamiento (parámetro de simulación, NO el resultado final)
BASE_RATE = {
    'control':              0.03,
    'newsletter':           0.06,
    'email_personalizado':  0.16,
    'descuento_agresivo':   0.22,
    'llamada_personal':     0.30,
}

# Efecto heterogéneo: clientes con recency alta (más inactivos) responden
# proporcionalmente mejor a ofertas agresivas (uplift real, no aditivo plano)
AGGRESSIVE_BOOST = {'descuento_agresivo': 0.10, 'llamada_personal': 0.15}

def simulate_response(row):
    base = BASE_RATE[row['treatment']]
    boost = AGGRESSIVE_BOOST.get(row['treatment'], 0.0) * max(row['recency_norm'], 0)
    freq_penalty = -0.03 * max(row['frequency_norm'], 0)  # clientes muy frecuentes ya están retenidos, menos margen de mejora
    prob = np.clip(base + boost + freq_penalty, 0.01, 0.95)
    return rng.binomial(1, prob)

ab_df['responded'] = ab_df.apply(simulate_response, axis=1)

print(f'⚠️  DATOS SINTÉTICOS — simulación de campaña histórica, no observaciones reales')
print(f'Clientes en el experimento: {len(ab_df):,}')
print(f'\nTasa de respuesta observada (simulada) por tratamiento:')
print(ab_df.groupby('treatment')['responded'].agg(['mean','count']).round(4))

### Modelo de uplift: T-learner

Entrenamos un modelo de respuesta independiente por cada tratamiento (incluyendo `control`) sobre las features RFM del cliente. Esto se llama **T-learner** (two-model learner): para predecir el `success_rate` de un cliente concreto bajo una acción concreta, usamos el modelo entrenado específicamente sobre los clientes que recibieron esa acción en el experimento simulado.

El resultado ya no es un número plano igual para todos (`success_rate = 0.18` para todo el mundo), sino una **probabilidad de respuesta condicionada al perfil de cada cliente** — el verdadero valor de tener datos de campaña en lugar de un benchmark fijo.

In [ ]:
from sklearn.linear_model import LogisticRegression

UPLIFT_FEATURES = ['recency', 'frequency', 'monetary', 'avg_order_value']

uplift_models = {}
for treatment in TREATMENTS:
    subset = ab_df[ab_df['treatment'] == treatment]
    X_t = np.log1p(subset[UPLIFT_FEATURES])
    y_t = subset['responded']
    model = LogisticRegression(max_iter=500, class_weight='balanced')
    model.fit(X_t, y_t)
    uplift_models[treatment] = model

print('✅ T-learner entrenado: un modelo de respuesta por cada tratamiento')
print(f'   Tratamientos: {list(uplift_models.keys())}')

# ── Predecir success_rate individual para TODOS los clientes y TODAS las acciones ─
X_all_uplift = np.log1p(df[UPLIFT_FEATURES])

predicted_success = pd.DataFrame({'Customer ID': df['Customer ID'].values})
for treatment in TREATMENTS:
    predicted_success[treatment] = uplift_models[treatment].predict_proba(X_all_uplift)[:, 1]

# Uplift = mejora frente al control (lo que realmente añade la acción de marketing)
for treatment in TREATMENTS:
    if treatment != 'control':
        predicted_success[f'uplift_{treatment}'] = predicted_success[treatment] - predicted_success['control']

print(f'\nDistribución de success_rate predicho por tratamiento (a nivel cliente):')
print(predicted_success[TREATMENTS].describe().round(3).to_string())

In [ ]:
# ── Visualización: el uplift varía por cliente, no es un número fijo ─────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for treatment, color in zip(
    ['email_personalizado','descuento_agresivo','llamada_personal'],
    [PALETTE['warning'], PALETTE['danger'], PALETTE['secondary']]
):
    axes[0].hist(predicted_success[f'uplift_{treatment}'], bins=40, alpha=0.55,
                 color=color, label=treatment)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Distribución del Uplift Predicho por Cliente', fontweight='bold')
axes[0].set_xlabel('Uplift (success_rate - control)')
axes[0].legend(fontsize=8)

# Uplift vs Recency — confirma que el modelo aprendió el patrón simulado
merged_viz = df[['Customer ID','recency']].merge(predicted_success, on='Customer ID')
axes[1].scatter(merged_viz['recency'], merged_viz['uplift_descuento_agresivo'],
                alpha=0.15, s=8, color=PALETTE['danger'])
axes[1].set_title('Uplift de Descuento Agresivo vs. Recency', fontweight='bold')
axes[1].set_xlabel('Recency (días)')
axes[1].set_ylabel('Uplift predicho')

plt.suptitle('El modelo de uplift aprende que clientes inactivos responden mejor a ofertas agresivas',
              fontsize=11, y=1.03)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'uplift_model_validation.png', bbox_inches='tight')
plt.show()

## 3 · Catálogo de costes de marketing

El `success_rate` por cliente y acción ya no es un número fijo — viene del modelo de uplift entrenado en la sección anterior (`predicted_success`). Aquí solo definimos los **costes**, que sí son supuestos de negocio razonables (lo que cuesta operativamente ejecutar cada canal):
- **Coste de contacto**: lo que cuesta llegar al cliente (email, llamada, etc.)
- **Coste del incentivo**: el descuento ofrecido, como % del CLV con un tope máximo

In [ ]:
# ── Catálogo de costes por acción ─────────────────────────────────────────────
# success_rate YA NO está aquí — se obtiene por cliente desde predicted_success
# (modelo de uplift de la sección 2). Esto es la mejora clave frente a un
# catálogo plano: cada cliente tiene su propia probabilidad de respuesta estimada.

ACTIONS = {
    'none': {
        'label':         'Sin acción',
        'contact_cost':  0.0,
        'incentive_pct': 0.0,
        'incentive_cap': 0.0,
    },
    'newsletter': {
        'label':         'Newsletter / Email genérico',
        'contact_cost':  0.05,
        'incentive_pct': 0.0,
        'incentive_cap': 0.0,
    },
    'email_personalizado': {
        'label':         'Email personalizado + descuento 10%',
        'contact_cost':  0.50,
        'incentive_pct': 0.10,
        'incentive_cap': 50.0,
    },
    'descuento_agresivo': {
        'label':         'Email + descuento 20%',
        'contact_cost':  0.50,
        'incentive_pct': 0.20,
        'incentive_cap': 150.0,
    },
    'llamada_personal': {
        'label':         'Llamada personal + oferta VIP',
        'contact_cost':  15.0,
        'incentive_pct': 0.15,
        'incentive_cap': 300.0,
    },
}

costs_df = pd.DataFrame(ACTIONS).T
costs_df.index.name = 'action_id'
print(costs_df.to_string())

## 4 · Cálculo de ROI esperado por acción y cliente

Para cada cliente y cada acción posible, calculamos:

$$\text{Valor esperado} = P(\text{churn}) \times \text{success\_rate}_{\text{cliente, acción}} \times \text{CLV}_{12m}$$

$$\text{Coste total} = \text{contact\_cost} + \min(\text{incentive\_pct} \times \text{CLV}_{12m},\ \text{incentive\_cap})$$

$$\text{ROI} = \frac{\text{Valor esperado} - \text{Coste total}}{\text{Coste total}}$$

La diferencia clave frente a un catálogo plano: `success_rate` ya no es el mismo número para todos los clientes — viene de `predicted_success`, el modelo de uplift entrenado en la sección 2. Un cliente con recency muy alta y la acción "descuento agresivo" tendrá un `success_rate` distinto (más alto) que un cliente reciente con la misma acción. La lógica de negocio de fondo no cambia: solo vale la pena actuar sobre un cliente que **podría irse** (churn_prob alto) y al que la acción concreta **puede salvar** (success_rate alto para ese cliente y esa acción). El valor salvado es proporcional a su CLV futuro.

In [ ]:
def compute_roi_table(df, actions, predicted_success):
    """Genera tabla larga cliente x acción con ROI esperado.
    success_rate se toma por cliente desde el modelo de uplift, no de un valor fijo.
    """
    df = df.merge(predicted_success, on='Customer ID', how='left', suffixes=('', '_pred'))
    rows = []
    for action_id, params in actions.items():
        incentive_cost = np.minimum(
            params['incentive_pct'] * df['clv_12m'],
            params['incentive_cap']
        )
        total_cost = params['contact_cost'] + incentive_cost

        # success_rate por cliente: columna del tratamiento correspondiente
        # ('control' para 'none', cada acción usa su propia columna del T-learner)
        treatment_col = 'control' if action_id == 'none' else action_id
        client_success_rate = df[treatment_col]

        expected_value_saved = df['churn_prob'] * client_success_rate * df['clv_12m']
        net_value = expected_value_saved - total_cost
        roi = np.where(total_cost > 0, net_value / total_cost, 0)

        rows.append(pd.DataFrame({
            'Customer ID':      df['Customer ID'].values,
            'action_id':        action_id,
            'action_label':     params['label'],
            'churn_prob':       df['churn_prob'].values,
            'clv_12m':          df['clv_12m'].values,
            'success_rate':     client_success_rate.values,
            'expected_value_saved': expected_value_saved.values,
            'total_cost':       total_cost.values,
            'net_value':        net_value.values,
            'roi':              roi,
        }))
    return pd.concat(rows, ignore_index=True)

roi_table = compute_roi_table(df, ACTIONS, predicted_success)
print(f'Filas (clientes x acciones): {len(roi_table):,}')
print(f'\n✅ success_rate ahora varía por cliente (no es un valor fijo de catálogo):')
print(roi_table.groupby('action_label')['success_rate'].describe()[['mean','std','min','max']].round(3))
roi_table.head(10)

In [ ]:
# ── ROI medio por acción (vista agregada) ─────────────────────────────────────
roi_summary = (
    roi_table.groupby('action_label')
    .agg(
        roi_medio        = ('roi', 'mean'),
        net_value_medio  = ('net_value', 'mean'),
        coste_medio      = ('total_cost', 'mean'),
        valor_salvado_medio = ('expected_value_saved', 'mean'),
    )
    .round(2)
    .sort_values('roi_medio', ascending=False)
)
print(roi_summary.to_string())

## 5 · Motor de decisión: asignación óptima de acción por cliente


In [ ]:
# ── Para cada cliente, elegir la acción con mayor net_value (no solo ROI) ────
# Usamos net_value en lugar de ROI puro para evitar elegir 'none' siempre
# (ROI de 'none' es indefinido/0, pero net_value de 'none' es siempre 0)

best_actions = (
    roi_table.sort_values('net_value', ascending=False)
    .groupby('Customer ID')
    .first()
    .reset_index()
)

decision = df.merge(
    best_actions[['Customer ID','action_id','action_label','expected_value_saved',
                  'total_cost','net_value','roi']],
    on='Customer ID', how='left'
)

print('── Distribución de Acciones Recomendadas ───────────────────')
print(decision['action_label'].value_counts().to_string())

print(f"\nNet value total esperado del plan: £{decision['net_value'].sum():,.0f}")
print(f"Coste total del plan:               £{decision['total_cost'].sum():,.0f}")

In [ ]:
# ── Visualización: acción recomendada por segmento de riesgo ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap riesgo x acción
if 'risk_segment' in decision.columns:
    cross = pd.crosstab(decision['risk_segment'], decision['action_label'])
    sns.heatmap(cross, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                linewidths=0.5, linecolor='white', cbar_kws={'label':'Nº clientes'})
    axes[0].set_title('Acción Recomendada por Segmento de Riesgo', fontweight='bold')
    axes[0].set_xlabel('Acción')
    axes[0].set_ylabel('Segmento de riesgo')
    axes[0].tick_params(axis='x', rotation=30)

# Net value por acción
net_by_action = decision.groupby('action_label')['net_value'].sum().sort_values()
colors = [PALETTE['danger'] if v < 0 else PALETTE['success'] for v in net_by_action.values]
axes[1].barh(net_by_action.index, net_by_action.values, color=colors, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Valor Neto Total Esperado por Acción (£)', fontweight='bold')
axes[1].set_xlabel('Net Value (£)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'action_assignment_overview.png', bbox_inches='tight')
plt.show()

## 6 · Simulación de presupuesto y priorización

En la práctica, marketing tiene un presupuesto limitado. Esta sección simula cómo asignarlo de forma óptima: ordenar todos los clientes por **net_value** (no por churn_prob ni CLV solos) y gastar el presupuesto en los de mayor retorno hasta que se agote.

In [ ]:
def simulate_budget(decision_df, budget):
    """Asigna presupuesto a clientes ordenados por net_value descendente."""
    sim = decision_df[decision_df['action_id'] != 'none'].copy()
    sim = sim.sort_values('net_value', ascending=False).reset_index(drop=True)
    sim['cum_cost'] = sim['total_cost'].cumsum()
    sim['funded']   = sim['cum_cost'] <= budget
    return sim

BUDGET_OPTIONS = [1000, 5000, 10000, 25000, 50000]

budget_results = []
for budget in BUDGET_OPTIONS:
    sim = simulate_budget(decision, budget)
    funded = sim[sim['funded']]
    budget_results.append({
        'budget':           budget,
        'clientes_cubiertos': len(funded),
        'coste_real':       funded['total_cost'].sum(),
        'valor_neto_total': funded['net_value'].sum(),
        'roi_global':       funded['net_value'].sum() / funded['total_cost'].sum() if funded['total_cost'].sum() > 0 else 0,
    })

budget_summary = pd.DataFrame(budget_results).round(2)
print(budget_summary.to_string(index=False))

In [ ]:
# ── Curva de retornos marginales decrecientes ─────────────────────────────────
sim_full = simulate_budget(decision, budget=decision['total_cost'].sum())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sim_full['cum_cost'], sim_full['net_value'].cumsum(),
             color=PALETTE['primary'], linewidth=2.5)
for budget in BUDGET_OPTIONS:
    axes[0].axvline(budget, linestyle=':', color=PALETTE['neutral'], alpha=0.5)
axes[0].set_title('Valor Neto Acumulado vs. Presupuesto Invertido', fontweight='bold')
axes[0].set_xlabel('Presupuesto acumulado (£)')
axes[0].set_ylabel('Valor neto acumulado (£)')

axes[1].bar(
    [str(b) for b in budget_summary['budget']],
    budget_summary['roi_global'],
    color=PALETTE['secondary'], edgecolor='white'
)
axes[1].set_title('ROI Global según Presupuesto Disponible', fontweight='bold')
axes[1].set_xlabel('Presupuesto (£)')
axes[1].set_ylabel('ROI')

plt.suptitle('Optimización de Presupuesto de Retención', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'budget_optimization.png', bbox_inches='tight')
plt.show()

print(f"\n💡 Insight: el ROI es más alto con presupuestos pequeños (se gastan primero")
print(f"   en los clientes de mayor retorno) y decrece a medida que se cubre más cartera.")

## 7 · Generación de mensajes personalizados

Para cada cliente con una acción recomendada, generamos el contenido del email/mensaje listo para enviar, personalizado con su segmento, su descuento calculado y un tono adecuado al nivel de riesgo.

In [ ]:
MESSAGE_TEMPLATES = {
    'newsletter': {
        'subject': 'Novedades que pueden interesarte',
        'body': (
            "Hola,\n\n"
            "Hemos preparado una selección de productos que creemos que te van a encantar, "
            "basados en tus compras anteriores.\n\n"
            "Échales un vistazo cuando tengas un momento.\n\n"
            "Un saludo,\nEl equipo"
        )
    },
    'email_personalizado': {
        'subject': 'Te echamos de menos — 10% de descuento solo para ti',
        'body': (
            "Hola,\n\n"
            "Hace tiempo que no te vemos por aquí y queríamos invitarte a volver con un "
            "descuento exclusivo del 10% en tu próximo pedido (hasta £{cap:.0f}).\n\n"
            "Usa el código BIENVENIDA10 en tu próxima compra.\n\n"
            "Un saludo,\nEl equipo"
        )
    },
    'descuento_agresivo': {
        'subject': 'Oferta especial: 20% de descuento — válida 7 días',
        'body': (
            "Hola,\n\n"
            "Queremos que vuelvas a confiar en nosotros. Por eso te ofrecemos un 20% de "
            "descuento en tu próxima compra (hasta £{cap:.0f}), válido durante los próximos 7 días.\n\n"
            "Código: VUELVE20\n\n"
            "Un saludo,\nEl equipo"
        )
    },
    'llamada_personal': {
        'subject': '[Guion de llamada — Cliente VIP en riesgo]',
        'body': (
            "GUION DE LLAMADA\n\n"
            "Cliente: {customer_id}\n"
            "Segmento: {segment}\n"
            "Gasto histórico: £{monetary:.0f}\n"
            "CLV estimado 12m: £{clv:.0f}\n\n"
            "Puntos a tratar:\n"
            "1. Agradecer su fidelidad como cliente histórico.\n"
            "2. Preguntar abiertamente si ha tenido algún problema con pedidos recientes.\n"
            "3. Ofrecer oferta VIP: 15% descuento (hasta £{cap:.0f}) + envío prioritario.\n"
            "4. Cerrar con fecha concreta de próximo contacto."
        )
    },
}

def generate_message(row):
    action_id = row['action_id']
    if action_id == 'none' or action_id not in MESSAGE_TEMPLATES:
        return None, None
    template = MESSAGE_TEMPLATES[action_id]
    incentive_cap = ACTIONS[action_id]['incentive_cap']
    body = template['body'].format(
        cap=incentive_cap,
        customer_id=row['Customer ID'],
        segment=row.get('cluster_label', 'N/A'),
        monetary=row.get('monetary', 0),
        clv=row.get('clv_12m', 0),
    )
    return template['subject'], body

decision[['message_subject','message_body']] = decision.apply(
    lambda r: pd.Series(generate_message(r)), axis=1
)

# Mostrar un ejemplo de cada tipo de mensaje
for action_id in ['email_personalizado', 'descuento_agresivo', 'llamada_personal']:
    sample = decision[decision['action_id'] == action_id].head(1)
    if len(sample) > 0:
        row = sample.iloc[0]
        print(f"{'='*60}\nACCIÓN: {row['action_label']} | Cliente: {row['Customer ID']}\n{'='*60}")
        print(f"Asunto: {row['message_subject']}\n")
        print(row['message_body'])
        print()

## 8 · Panel ejecutivo de impacto esperado


In [ ]:
actionable = decision[decision['action_id'] != 'none']

print('='*70)
print(' PLAN DE RETENCIÓN — RESUMEN EJECUTIVO')
print('='*70)
print(f"\nClientes totales analizados:      {len(decision):,}")
print(f"Clientes con acción recomendada:  {len(actionable):,} ({len(actionable)/len(decision)*100:.1f}%)")
print(f"Clientes sin acción (bajo riesgo):{(decision['action_id']=='none').sum():,}")

print(f"\nInversión total recomendada:      £{actionable['total_cost'].sum():,.0f}")
print(f"Valor esperado salvado:           £{actionable['expected_value_saved'].sum():,.0f}")
print(f"Valor neto esperado (ROI £):      £{actionable['net_value'].sum():,.0f}")
print(f"ROI global del plan:              {actionable['net_value'].sum()/actionable['total_cost'].sum()*100:.0f}%")

print(f"\n── Desglose por acción ──────────────────────────────────────")
breakdown = (
    actionable.groupby('action_label')
    .agg(n=('Customer ID','count'), coste=('total_cost','sum'),
         valor_neto=('net_value','sum'))
    .round(0)
    .sort_values('valor_neto', ascending=False)
)
print(breakdown.to_string())
print('\n' + '='*70)

In [ ]:
# ── Waterfall del impacto financiero ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5.5))

values = [
    ('CLV en riesgo\n(churn x CLV)', (decision['churn_prob']*decision['clv_12m']).sum(), PALETTE['neutral']),
    ('Valor salvado\nesperado',       actionable['expected_value_saved'].sum(),         PALETTE['success']),
    ('Coste del plan',                -actionable['total_cost'].sum(),                  PALETTE['danger']),
    ('Valor neto\nfinal',             actionable['net_value'].sum(),                    PALETTE['primary']),
]

labels  = [v[0] for v in values]
amounts = [v[1] for v in values]
colors  = [v[2] for v in values]

ax.bar(labels, amounts, color=colors, edgecolor='white', width=0.6)
for i, v in enumerate(amounts):
    ax.text(i, v + (max(amounts)*0.02 if v>=0 else min(amounts)*0.05),
            f'£{v:,.0f}', ha='center', fontweight='bold', fontsize=10)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Impacto Financiero Esperado del Plan de Retención', fontweight='bold', fontsize=13)
ax.set_ylabel('£')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'financial_impact_waterfall.png', bbox_inches='tight')
plt.show()

## 9 · Exportar plan de acción


In [ ]:
# ── Tabla final de decisión por cliente ───────────────────────────────────────
export_cols = ['Customer ID']
for c in ['cluster_label','RFM_segment','risk_segment','recency','frequency',
          'monetary','clv_12m','churn_prob','action_id','action_label',
          'expected_value_saved','total_cost','net_value','roi',
          'message_subject','message_body']:
    if c in decision.columns:
        export_cols.append(c)

marketing_plan = decision[export_cols].copy()
marketing_plan = marketing_plan.sort_values('net_value', ascending=False)

marketing_plan.to_parquet(DATA_DIR / 'marketing_plan.parquet', index=False)
marketing_plan.drop(columns=['message_body']).to_csv(
    OUTPUT_DIR / 'marketing_plan_summary.csv', index=False
)

# CSV separado solo con mensajes completos (para copiar/pegar en herramienta de email)
messages_export = marketing_plan[
    marketing_plan['action_id'] != 'none'
][['Customer ID','action_label','message_subject','message_body']]
messages_export.to_csv(OUTPUT_DIR / 'messages_ready_to_send.csv', index=False)

# Guardar catálogo de acciones y parámetros
with open(OUTPUT_DIR / 'action_catalog.json', 'w', encoding='utf-8') as f:
    json.dump(ACTIONS, f, indent=2, ensure_ascii=False)

print(f'✅ marketing_plan.parquet guardado ({len(marketing_plan):,} clientes)')
print(f'✅ marketing_plan_summary.csv guardado')
print(f'✅ messages_ready_to_send.csv guardado ({len(messages_export):,} mensajes)')
print(f'✅ action_catalog.json guardado')

print('\n📦 Archivos en outputs/06_marketing/:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'   {f.name:<40} {f.stat().st_size/1024:>7.1f} KB')

print('\n✅ Notebook 06 completado.')

---
## Conclusiones

Este notebook traduce probabilidades de churn y CLV en decisiones financieras concretas. La pregunta ya no es "¿qué clientes van a irse?" sino "¿en cuáles merece la pena invertir, cuánto, y con qué acción exactamente?"

La pieza central es el **modelo de uplift**: en lugar de asumir que todos los clientes responden igual a una misma oferta, se entrena un modelo independiente por tratamiento (T-learner) que aprende cómo el efecto de cada acción varía según el perfil RFM del cliente. El resultado es que un cliente inactivo de hace meses y uno que compró ayer pueden recibir la misma acción pero con un `success_rate` distinto, reflejando que el primero tiene más margen de mejora.

El motor de decisión no asigna la acción más agresiva a todo el mundo: el ROI de cada acción depende del producto entre `churn_prob`, `success_rate` personalizado y CLV, menos el coste. La simulación de presupuesto muestra que los retornos son decrecientes — los primeros euros invertidos son los más rentables porque se dirigen a los clientes de mayor ROI primero.

Los mensajes generados están listos para que un equipo de marketing los revise y envíe, eliminando la fricción entre el modelo predictivo y la acción real.